# Part 2 — NeSy fine-tune (predicates → FSM safety shield → logic-shaped reward)

Take the **frozen Part-1 PPO policy** and make it obey the temporal-logic traffic rules in two layers: a **zero-retraining FSM safety shield** (hard constraints) and a **logic-shaped reward fine-tune** (soft heuristics). Re-run the Part-1 harness on the same seeds **plus a per-rule violation rate**, counted by an independent monitor (not the shield/reward the agent sees). Record a video of the shielded+fine-tuned agent.

Covers IMPLEMENTATION_PLAN.md **Part 2** (Steps A–C). Run `colab_1_baseline.ipynb` first.

## 1. Mount Drive + load the GitHub token (private repo)

In [ ]:
import os, warnings, logging
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'.*Gym has been unmaintained.*')
logging.getLogger('gym').setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

## 2. Bootstrap the repo

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=nesy-highway-driving
curl -fsSL -H "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving
from utils import load_config, drive_path, save_json
cfg = load_config('configs/highway.yaml')

## 3. Load the frozen Part-1 PPO baseline

In [ ]:
from agents.baselines import load_model
baseline = load_model(drive_path(cfg, 'checkpoints', 'ppo.zip'), 'ppo')

## 4. Step A — predicates (perception → logic)
Sanity-check the grounded predicates on one live scene before trusting any downstream number.

In [ ]:
from envs.highway_factory import make_env, read_scene
from nesy.roadmap import predicates

env = make_env(cfg)
obs, _ = env.reset(seed=cfg['eval_seeds'][0])
for _ in range(5):
    a, _ = baseline.predict(obs, deterministic=True)
    obs, *_ = env.step(a)
print(predicates(read_scene(env), cfg))
env.close()

## 5. Step B — FSM safety shield (hard constraints, zero retraining)
Evaluate the same frozen policy **with** the shield: a clean on/off ablation on identical seeds, with per-rule violations.

In [ ]:
from eval.evaluate import evaluate, record_video

m_baseline = evaluate(baseline, cfg, count_violations=True)
m_shield   = evaluate(baseline, cfg, apply_shield=True, count_violations=True)
save_json(m_baseline, drive_path(cfg, 'metrics', 'p2_baseline.json'))
save_json(m_shield,   drive_path(cfg, 'metrics', 'p2_shield.json'))
print('shield violation rates:', m_shield['summary']['rule_violation_rate'])

## 6. Step C — logic-shaped reward fine-tune (soft heuristics)
Warm-start the frozen policy, continue training on `reward - Σ λ_i·violation_i` (lower LR, fewer steps), then evaluate the fine-tuned policy with the shield on.

In [ ]:
from agents.baselines import finetune_logic_reward

nesy_model = finetune_logic_reward(load_model(drive_path(cfg, 'checkpoints', 'ppo.zip'), 'ppo'), cfg)
m_reward          = evaluate(nesy_model, cfg, apply_shield=False, count_violations=True)
m_shield_reward   = evaluate(nesy_model, cfg, apply_shield=True,  count_violations=True)
save_json(m_reward,        drive_path(cfg, 'metrics', 'p2_reward.json'))
save_json(m_shield_reward, drive_path(cfg, 'metrics', 'p2_shield_reward.json'))

## 7. Four-config comparison table (with per-rule violations)

In [ ]:
import pandas as pd

def row(name, m):
    s = m['summary']; v = s.get('rule_violation_rate', {})
    out = {'config': name, 'crash': round(s['crash_rate'], 3),
           'on_road_%': round(s['on_road_pct']['mean'], 1),
           'overtakes': round(s['overtakes']['mean'], 2),
           'return': round(s['return']['mean'], 2)}
    for k in ('RG1', 'RG3', 'RI1', 'RI2', 'RG4'):
        out[k] = round(v.get(k, float('nan')), 3)
    return out

pd.DataFrame([
    row('Baseline (P1)',   m_baseline),
    row('+ FSM shield',    m_shield),
    row('+ Logic reward',  m_reward),
    row('+ Shield+reward', m_shield_reward),
])

## 8. Record a video of the shielded + fine-tuned agent

In [ ]:
nesy_video = record_video(nesy_model, cfg, drive_path(cfg, 'videos', 'ppo_nesy.mp4'), apply_shield=True)
print('NeSy video dir:', nesy_video)

**Exit criterion (headline):** the shielded + fine-tuned policy achieves equal-or-better overtaking with **provably fewer rule violations** than the pure-neural baseline (compare the `RG*/RI*` columns). Next: `colab_3_metadrive_race.ipynb`.